# Diwali E-commerce Sales Analysis
**Dataset:** Synthetic practice dataset created for this assignment (60 order records).  
**Important:** The data is simulated, not real company/customer data. Use the actual results produced by running the notebook in Part F.


## Setup and load dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("diwali_ecommerce_dataset.csv")
df.head()


## Part A – Dataset creation and validation

In [ ]:
print("Shape:", df.shape)
print("Cities:", df["City"].nunique(), sorted(df["City"].unique()))
print("Categories:", df["Category"].nunique(), sorted(df["Category"].unique()))
print("Payment methods:", sorted(df["Payment_Mode"].unique()))
df.to_csv("diwali_ecommerce_dataset.csv", index=False)


## Part B – Python fundamentals and functions

In [ ]:
def final_price(price, discount):
    return price * (1 - discount / 100)

def total_bill(quantity, price, discount=0):
    return quantity * final_price(price, discount)

def age_group(age):
    if age < 20:
        return "Teenager"
    elif age < 60:
        return "Adult"
    return "Senior Citizen"

def highest_price(prices):
    return max(prices)

def spending_category(spending):
    if spending < 2000:
        return "Low Spender"
    elif spending <= 5000:
        return "Medium Spender"
    return "High Spender"

def average_rating(ratings):
    return sum(ratings) / len(ratings) if ratings else 0

def eligible_for_extra_discount(total_spending):
    return total_spending >= 5000

# Demonstrations
print("Final price:", final_price(1000, 10))
print("Total bill:", total_bill(3, 1000, 10))
print("Age group:", age_group(24))
print("Highest price:", highest_price([500, 1200, 900]))
print("Spending category:", spending_category(4500))
print("Average rating:", average_rating([4, 5, 3, 4]))
print("Extra discount eligible:", eligible_for_extra_discount(6000))


## Part C – NumPy challenges

In [ ]:
prices = np.array([499, 799, 999, 1299, 1599, 1999, 2499, 2999, 3499, 3999, 4499, 4999, 5999, 6999, 7999])
print("Maximum:", np.max(prices))
print("Minimum:", np.min(prices))
print("Mean:", np.mean(prices))
print("Median:", np.median(prices))
print("Standard deviation:", np.std(prices))
print("Prices above average:", prices[prices > np.mean(prices)])
print("Reversed:", prices[::-1])
print("Ascending:", np.sort(prices))
print("Descending:", np.sort(prices)[::-1])

daily_sales = np.array([12000, 15000, 11000, 18000, 22000, 17000, 25000, 21000, 19000, 28000])
print("Daily sales total:", daily_sales.sum())
print("Daily sales average:", daily_sales.mean())
print("Maximum daily sales:", daily_sales.max())

category_a = np.array([1000, 1200, 900, 1500, 1700])
category_b = np.array([800, 1100, 1300, 1000, 1600])
print("Element-wise category sales addition:", category_a + category_b)

random_sales = np.random.default_rng(42).integers(1000, 10000, 15)
print("Random sales:", random_sales)
print("Top 5 sales:", np.sort(random_sales)[-5:][::-1])


## Part D – Pandas data analysis

In [ ]:
# Basic exploration
display(df.head(10))
display(df.tail(5))
print("Rows and columns:", df.shape)
print("Column names:", df.columns.tolist())
print("Data types:\n", df.dtypes)
display(df.describe(include="all"))
print("Unique customers:", df["Customer_ID"].nunique())
print("Total orders:", df["Order_ID"].nunique())

# Cleaning
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df = df.dropna(subset=["Order_Date", "Price", "Quantity", "Discount"])
df["Total_Amount"] = df["Quantity"] * df["Price"] * (1 - df["Discount"] / 100)
df["Age_Group"] = pd.cut(df["Age"], bins=[0,19,59,120],
                         labels=["Teenager","Adult","Senior Citizen"])
display(df.head())

# Required business analysis
print("Top 5 purchased products (quantity):")
display(df.groupby("Product_Name")["Quantity"].sum().sort_values(ascending=False).head(5))
print("Total revenue:", round(df["Total_Amount"].sum(), 2))
print("Highest revenue city:", df.groupby("City")["Total_Amount"].sum().idxmax())
print("Preferred payment method:", df["Payment_Mode"].value_counts().idxmax())
print("Average customer rating:", round(df["Rating"].mean(), 2))
print("Customer with maximum total spending:")
customer_spend = df.groupby(["Customer_ID","Customer_Name","City"])["Total_Amount"].sum().sort_values(ascending=False)
display(customer_spend.head(1))
print("City-wise sales:")
display(df.groupby("City")["Total_Amount"].sum().sort_values(ascending=False))
print("Category-wise revenue:")
display(df.groupby("Category")["Total_Amount"].sum().sort_values(ascending=False))
print("Average spending per customer:", round(df.groupby("Customer_ID")["Total_Amount"].sum().mean(), 2))
print("Customers spending more than ₹5,000:")
display(df.groupby(["Customer_ID","Customer_Name"])["Total_Amount"].sum().loc[lambda x: x > 5000].sort_values(ascending=False))
print("Category with highest quantity sold:", df.groupby("Category")["Quantity"].sum().idxmax())
print("Total discount amount (₹):", round((df["Quantity"]*df["Price"]*df["Discount"]/100).sum(),2))
print("Orders by date:")
display(df.groupby(df["Order_Date"].dt.date)["Order_ID"].nunique())
print("Top 5 customers by spending:")
display(df.groupby(["Customer_ID","Customer_Name"])["Total_Amount"].sum().sort_values(ascending=False).head(5))
print("Average age by category:")
display(df.groupby("Category", observed=False)["Age"].mean().sort_values(ascending=False))
print("Product with lowest sales revenue:")
display(df.groupby("Product_Name")["Total_Amount"].sum().sort_values().head(1))


## Part E – Matplotlib and Seaborn visualizations

In [ ]:
# 1. Category-wise revenue
plt.figure(figsize=(9,5))
sns.barplot(data=df, x="Category", y="Total_Amount", estimator=sum, errorbar=None)
plt.title("Category-wise Revenue"); plt.xlabel("Category"); plt.ylabel("Revenue (₹)")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

# 2. Payment method distribution
plt.figure(figsize=(7,7))
df["Payment_Mode"].value_counts().plot.pie(autopct="%1.1f%%", ylabel="")
plt.title("Payment Method Distribution"); plt.tight_layout(); plt.show()

# 3. Customer age distribution
plt.figure(figsize=(8,5))
sns.histplot(data=df, x="Age", bins=10, kde=True)
plt.title("Customer Age Distribution"); plt.xlabel("Age"); plt.ylabel("Number of Customers")
plt.tight_layout(); plt.show()

# 4. Order count by city
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="City", order=df["City"].value_counts().index)
plt.title("Number of Orders by City"); plt.xlabel("City"); plt.ylabel("Number of Orders")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

# 5. Product price distribution
plt.figure(figsize=(8,5))
sns.boxplot(data=df, y="Price")
plt.title("Distribution of Product Prices"); plt.ylabel("Price (₹)")
plt.tight_layout(); plt.show()

# 6. Daily sales trend
daily = df.groupby("Order_Date")["Total_Amount"].sum().reset_index()
plt.figure(figsize=(10,5))
sns.lineplot(data=daily, x="Order_Date", y="Total_Amount", marker="o")
plt.title("Daily Sales Trend"); plt.xlabel("Order Date"); plt.ylabel("Sales (₹)")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# 7. Correlation heatmap
plt.figure(figsize=(8,5))
sns.heatmap(df[["Age","Quantity","Price","Discount","Rating","Total_Amount"]].corr(),
            annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numerical Columns"); plt.tight_layout(); plt.show()

# 8. Quantity vs total amount
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Quantity", y="Total_Amount", hue="Category")
plt.title("Quantity Purchased vs Total Amount"); plt.xlabel("Quantity"); plt.ylabel("Total Amount (₹)")
plt.tight_layout(); plt.show()

# 9. Top 5 products by quantity
top_products = df.groupby("Product_Name")["Quantity"].sum().nlargest(5).reset_index()
plt.figure(figsize=(8,5))
sns.barplot(data=top_products, x="Quantity", y="Product_Name")
plt.title("Top 5 Most Purchased Products"); plt.xlabel("Quantity Sold"); plt.ylabel("Product")
plt.tight_layout(); plt.show()

# 10. Additional visualization: revenue by city
plt.figure(figsize=(8,5))
city_revenue = df.groupby("City")["Total_Amount"].sum().sort_values(ascending=False).reset_index()
sns.barplot(data=city_revenue, x="City", y="Total_Amount")
plt.title("Revenue by City"); plt.xlabel("City"); plt.ylabel("Revenue (₹)")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()


## Part F – Business insights and data storytelling

In [ ]:
# Generate answers directly from the dataset
city_sales = df.groupby("City")["Total_Amount"].sum().sort_values(ascending=False)
category_sales = df.groupby("Category")["Total_Amount"].sum().sort_values(ascending=False)
payment_counts = df["Payment_Mode"].value_counts()
customer_totals = df.groupby(["Customer_ID","Customer_Name","City"])["Total_Amount"].sum().sort_values(ascending=False)
age_sales = df.groupby("Age_Group", observed=False)["Total_Amount"].sum().sort_values(ascending=False)
category_discount = df.groupby("Category")["Discount"].mean().sort_values(ascending=False)
category_avg_order = df.groupby("Category")["Total_Amount"].mean().sort_values(ascending=False)

print("1. City contributing most revenue:", city_sales.index[0], "₹", round(city_sales.iloc[0],2))
print("2. Best-performing category by revenue:", category_sales.index[0], "₹", round(category_sales.iloc[0],2))
print("3. Most preferred payment method:", payment_counts.index[0], "(", payment_counts.iloc[0], "orders )")
print("4. Highest-spending customer:", customer_totals.index[0][1], "₹", round(customer_totals.iloc[0],2))
print("5. Average customer spending:", round(df.groupby("Customer_ID")["Total_Amount"].sum().mean(),2))
print("6. Category needing improvement (lowest revenue):", category_sales.index[-1])
print("7. Average discount by category:\n", category_discount)
print("8. Age group contributing most sales:", age_sales.index[0])
print("9. Recommendations: prioritize inventory for the highest-revenue category; target the leading city with campaigns; offer bundles to improve lower-revenue categories; optimize discounts using category-level performance; promote popular payment options.")
print("10. Management decisions: allocate stock based on product demand; focus marketing on high-revenue cities and age groups; test targeted discounts and monitor conversion/revenue.")
print("\nHighest average discount category:", category_discount.index[0])
print("Highest revenue category:", category_sales.index[0])


## Part G – Optional challenges

In [ ]:
# Challenge 1: Diwali Shopping Champion
champion_id = customer_totals.index[0][0]
champion = df[df["Customer_ID"] == champion_id]
print("Customer:", champion["Customer_Name"].iloc[0])
print("City:", champion["City"].mode().iloc[0])
print("Number of orders:", champion["Order_ID"].nunique())
print("Total spending:", round(champion["Total_Amount"].sum(),2))
print("Most purchased category:", champion.groupby("Category")["Quantity"].sum().idxmax())

# Challenge 2: Best Festive Deal
print("Average discount by category:\n", df.groupby("Category")["Discount"].mean().sort_values(ascending=False))
print("Revenue by category:\n", category_sales)

# Challenge 3: Three additional business questions
print("Q1: Which gender generated the highest revenue?\n", df.groupby("Gender")["Total_Amount"].sum().sort_values(ascending=False))
print("Q2: Which weekday had the highest sales?\n", df.assign(Weekday=df["Order_Date"].dt.day_name()).groupby("Weekday")["Total_Amount"].sum().sort_values(ascending=False))
print("Q3: Which payment method had the highest average order value?\n", df.groupby("Payment_Mode")["Total_Amount"].mean().sort_values(ascending=False))

# Challenge 4: Mini dashboard with three charts
fig, axes = plt.subplots(1, 3, figsize=(18,5))
category_sales.plot(kind="bar", ax=axes[0], title="Revenue by Category", ylabel="₹")
payment_counts.plot(kind="pie", ax=axes[1], autopct="%1.0f%%", title="Payment Mix", ylabel="")
city_sales.plot(kind="bar", ax=axes[2], title="Revenue by City", ylabel="₹")
plt.tight_layout()
plt.show()


## Final conclusion
After running the notebook, write a short conclusion using the actual output values. Mention the leading city, best-performing category, preferred payment method, highest-spending customer, and two or three actionable recommendations. Do not present simulated dataset findings as real company results.
